# Open Field Test — Figure Replication (Figures 5 & 6)

**Publication:** [Mantas et al. (2024) — bioRxiv](https://doi.org/10.1101/2024.12.22.629963)  
**Dataset:** Meletis Lab, Karolinska Institutet

This notebook replicates panels from Figures 5 and 6 of the manuscript using data
stored in NWB format. We use a representative subset of 28 sessions (2 per experimental
group) that have been re-converted with VAME motif data.

### Replicable panels

| Panel | Description | Status |
|-------|-------------|--------|
| **Fig 5B** | FP signal vs speed correlation | Partial (subset of sessions) |
| **Fig 5C** | DLC pose + VAME motif examples | Yes |
| **Fig 5D** | Motif duration and speed distributions | Yes |
| **Fig 5E** | Heatmap: z-scored FP aligned to motif onset | Partial (FP sessions only) |
| **Fig 5H** | FP AUC vs mean motif speed | Partial |
| **Fig 6B** | Locomotor trajectories per group | Yes |
| **Fig 6C** | Speed / angular speed box plots per group | Yes |
| **Fig 6I** | Normalized motif usage across groups | Yes |

### Not replicable from NWB alone
- Fig 5A, 6A: Schematics (illustrations)
- Fig 6D-F: PCA of motif usage (needs full cohort + VAME model metadata)
- Fig 6J-K: Classifier (needs sklearn, full cohort)

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from pynwb import NWBHDF5IO
from scipy import stats

%matplotlib inline
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 9

## Load all NWB sessions

We load metadata and key data arrays from the 28 representative sessions.
Each session has: pose estimation, kinematics, VAME motifs, and optionally fiber photometry.

In [2]:
nwb_dir = Path("../../../nwb_output/open_field_test")

# The 28 re-converted sessions with VAME
target_sessions = [
    "sub-543453_ses-tmaze_2022-06-18T12_07_43",
    "sub-544684_ses-tmaze_2022-06-18T13_45_10",
    "sub-542163_ses-tmaze_2022-06-18T12_23_45",
    "sub-542159_ses-tmaze_2022-06-18T12_39_07",
    "sub-687662_ses-tmaze_2023-05-05T13_09_53",
    "sub-687642_ses-tmaze_2023-05-16T12_21_10",
    "sub-661507_ses-tmaze_2023-05-05T12_07_03",
    "sub-661506_ses-tmaze_2023-05-16T11_53_31",
    "sub-776769_ses-oft_2024-03-05T12_11_33",
    "sub-776770_ses-oft_2024-03-05T12_45_35",
    "sub-796287_ses-oft_2024-04-10T10_38_05",
    "sub-802372_ses-oft_2024-04-11T09_03_40",
    "sub-698644_ses-tmaze_2023-05-25T17_09_12",
    "sub-700067_ses-tmaze_2023-05-25T17_27_56",
    "sub-700065_ses-tmaze_2023-05-25T17_19_25",
    "sub-700068_ses-tmaze_2023-05-25T17_36_17",
    "sub-717250_ses-tmaze_2023-09-06T14_55_02",
    "sub-717252_ses-tmaze_2023-09-06T15_02_31",
    "sub-721924_ses-tmaze_2023-09-13T13_37_26",
    "sub-721925_ses-tmaze_2023-09-13T13_30_35",
    "sub-687643_ses-oft_2023-05-16T15_31_06",
    "sub-687644_ses-oft_2023-05-16T15_40_21",
    "sub-714027_ses-tmaze_2023-09-21T08_38_08",
    "sub-714028_ses-tmaze_2023-09-21T08_45_06",
    "sub-914943_ses-oft_2025-05-06T15_48_52",
    "sub-914945_ses-oft_2025-05-06T15_56_53",
    "sub-960161_ses-tmaze_2025-07-16T11_57_06",
    "sub-960165_ses-tmaze_2025-07-16T12_05_10",
]

sessions = []
for name in target_sessions:
    path = nwb_dir / f"{name}.nwb"
    if not path.exists():
        continue
    io = NWBHDF5IO(str(path), mode="r")
    nwb = io.read()
    behavior = nwb.processing["behavior"]

    # Extract key data
    speed = behavior["Kinematics"].time_series["speed"].data[:]
    angular_speed = behavior["Kinematics"].time_series["angular_speed"].data[:]

    # Pose (snout trajectory)
    pose = behavior["PoseEstimationDeepLabCut"]
    snout_xy = pose.pose_estimation_series["PoseEstimationSeriesSnout"].data[:]

    # VAME motifs
    vame_data = None
    if "VAMEMotifs157" in behavior.data_interfaces:
        vame_data = behavior["VAMEMotifs157"].data[:]

    # Fiber photometry (optional)
    fp_signal = None
    fp_timestamps = None
    if "FiberPhotometryResponseSeries" in nwb.acquisition:
        fp = nwb.acquisition["FiberPhotometryResponseSeries"]
        fp_signal = fp.data[:]
        fp_timestamps = fp.timestamps[:]

    session_info = dict(
        name=name,
        session_id=nwb.session_id,
        subject_id=nwb.subject.subject_id,
        genotype=nwb.subject.genotype,
        speed=speed,
        angular_speed=angular_speed,
        snout_xy=snout_xy,
        vame=vame_data,
        fp_signal=fp_signal,
        fp_timestamps=fp_timestamps,
        io=io,
    )

    # Parse experiment/group from session_id
    session_info["experiment"] = nwb.session_id
    sessions.append(session_info)

print(f"Loaded {len(sessions)} sessions")
print(f"Sessions with FP: {sum(1 for s in sessions if s['fp_signal'] is not None)}")
print(f"Sessions with VAME: {sum(1 for s in sessions if s['vame'] is not None)}")

Loaded 0 sessions
Sessions with FP: 0
Sessions with VAME: 0


## Figure 5B — FP signal vs locomotor speed correlation

For sessions with fiber photometry, we compute the Pearson correlation between the
dLight/GCaMP signal and linear speed. The FP signal is downsampled from ~60 Hz to 30 Hz
to match the kinematics rate.

In [3]:
fp_sessions = [s for s in sessions if s["fp_signal"] is not None]

if fp_sessions:
    fig, axes = plt.subplots(1, len(fp_sessions), figsize=(4 * len(fp_sessions), 4), squeeze=False)

    for i, s in enumerate(fp_sessions):
        ax = axes[0, i]
        # Downsample FP to 30 Hz by taking every other sample (~60 Hz -> ~30 Hz)
        fp_ds = s["fp_signal"][::2]
        speed = s["speed"]
        n = min(len(fp_ds), len(speed))
        fp_ds = fp_ds[:n]
        speed_trimmed = speed[:n]

        r, p = stats.pearsonr(fp_ds, speed_trimmed)
        ax.scatter(speed_trimmed[::10], fp_ds[::10], s=1, alpha=0.3, c="teal")
        ax.set_xlabel("Speed (pixels/s)")
        ax.set_ylabel("FP signal (a.u.)")
        ax.set_title(f"{s['subject_id']}\nr={r:.3f}, p={p:.1e}")

    plt.suptitle("Fig 5B: FP signal vs locomotor speed", fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()

    # Summary box plot of correlations
    correlations = []
    for s in fp_sessions:
        fp_ds = s["fp_signal"][::2]
        n = min(len(fp_ds), len(s["speed"]))
        r, _ = stats.pearsonr(fp_ds[:n], s["speed"][:n])
        correlations.append(r)

    fig, ax = plt.subplots(figsize=(3, 4))
    ax.boxplot(correlations, widths=0.5)
    ax.scatter(np.ones(len(correlations)), correlations, c="teal", zorder=5, s=40)
    ax.axhline(0, color="gray", linestyle="--", linewidth=0.5)
    ax.set_ylabel("Pearson r (FP signal vs speed)")
    ax.set_title("Fig 5B: FP–speed correlation", fontweight="bold")
    ax.set_xticks([1])
    ax.set_xticklabels(["FP sessions"])
    plt.tight_layout()
    plt.show()
else:
    print("No sessions with fiber photometry in this subset.")

No sessions with fiber photometry in this subset.


## Figure 5C — DLC pose estimation + VAME motif ethogram

Left: Example pose estimation showing 6 keypoints tracked by DeepLabCut.  
Right: VAME behavioral motif sequence for a representative segment.

In [4]:
# Pick a session with VAME
example = next(s for s in sessions if s["vame"] is not None)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: DLC trajectory (first 60s)
ax = axes[0]
n_frames_60s = 30 * 60
xy = example["snout_xy"][:n_frames_60s]
t = np.arange(len(xy)) / 30.0
sc = ax.scatter(xy[:, 0], xy[:, 1], c=t, s=0.5, cmap="viridis", alpha=0.6)
ax.set_xlabel("x (pixels)")
ax.set_ylabel("y (pixels)")
ax.set_title(f"DLC snout trajectory (first 60s)\n{example['subject_id']}")
ax.set_aspect("equal")
plt.colorbar(sc, ax=ax, label="Time (s)")

# Right: VAME motif ethogram (first 60s)
ax = axes[1]
motifs = example["vame"][:n_frames_60s]
t_motifs = np.arange(len(motifs)) / 30.0
ax.scatter(t_motifs, motifs, c=motifs, s=0.3, cmap="tab20", alpha=0.7)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Motif ID")
ax.set_title("VAME motif ethogram (first 60s)")

plt.suptitle("Fig 5C: Pose estimation and behavioral motifs", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

StopIteration: 

## Figure 5D — Motif duration and speed distributions

For each VAME motif, we compute:
- **Duration**: Mean duration of consecutive runs of each motif
- **Speed**: Mean body-center speed during each motif

Motifs are ordered by increasing mean speed.

In [ ]:
# Aggregate motif stats across all sessions with VAME
motif_speeds = {}  # motif_id -> list of mean speeds
motif_durations = {}  # motif_id -> list of bout durations in seconds

for s in sessions:
    if s["vame"] is None:
        continue
    motifs = s["vame"]
    speed = s["speed"]
    n = min(len(motifs), len(speed))
    motifs = motifs[:n]
    speed = speed[:n]

    # Find bouts (consecutive runs of same motif)
    changes = np.where(np.diff(motifs) != 0)[0] + 1
    bout_starts = np.concatenate([[0], changes])
    bout_ends = np.concatenate([changes, [n]])

    for start, end in zip(bout_starts, bout_ends):
        mid = motifs[start]
        dur = (end - start) / 30.0  # seconds
        mean_speed = speed[start:end].mean()

        motif_speeds.setdefault(mid, []).append(mean_speed)
        motif_durations.setdefault(mid, []).append(dur)

# Order motifs by mean speed
motif_ids = sorted(motif_speeds.keys())
mean_speed_per_motif = {m: np.mean(motif_speeds[m]) for m in motif_ids}
motif_ids_sorted = sorted(motif_ids, key=lambda m: mean_speed_per_motif[m])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Duration distribution
ax = axes[0]
dur_medians = [np.median(motif_durations[m]) for m in motif_ids_sorted]
dur_q25 = [np.percentile(motif_durations[m], 25) for m in motif_ids_sorted]
dur_q75 = [np.percentile(motif_durations[m], 75) for m in motif_ids_sorted]
y_pos = np.arange(len(motif_ids_sorted))
ax.barh(y_pos, dur_medians, xerr=[np.array(dur_medians) - np.array(dur_q25),
         np.array(dur_q75) - np.array(dur_medians)], height=0.7, color="steelblue", alpha=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(motif_ids_sorted, fontsize=6)
ax.set_xlabel("Motif duration (s)")
ax.set_ylabel("Motif ID (ordered by speed)")
ax.set_title("Motif duration")

# Speed distribution
ax = axes[1]
speed_medians = [np.median(motif_speeds[m]) for m in motif_ids_sorted]
speed_q25 = [np.percentile(motif_speeds[m], 25) for m in motif_ids_sorted]
speed_q75 = [np.percentile(motif_speeds[m], 75) for m in motif_ids_sorted]
ax.barh(y_pos, speed_medians, xerr=[np.array(speed_medians) - np.array(speed_q25),
         np.array(speed_q75) - np.array(speed_medians)], height=0.7, color="darkorange", alpha=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(motif_ids_sorted, fontsize=6)
ax.set_xlabel("Speed (pixels/s)")
ax.set_ylabel("Motif ID (ordered by speed)")
ax.set_title("Mean motif speed")

plt.suptitle("Fig 5D: Motif duration and speed distributions", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print(f"Total motifs: {len(motif_ids_sorted)}")
print(f"Speed range: {min(speed_medians):.1f} – {max(speed_medians):.1f} pixels/s")

## Figure 5E — Heatmap: z-scored FP aligned to motif onset

For sessions with fiber photometry, we align the z-scored FP signal to the onset of each
VAME motif bout. Motifs are ordered by increasing mean speed (same order as Fig 5D).

In [ ]:
fp_vame_sessions = [s for s in sessions if s["fp_signal"] is not None and s["vame"] is not None]

if fp_vame_sessions:
    # Collect motif-triggered FP averages
    window_pre = 30  # 1s before onset (at 30 Hz)
    window_post = 60  # 2s after onset
    motif_triggered = {m: [] for m in motif_ids_sorted}

    for s in fp_vame_sessions:
        # Downsample FP to 30 Hz
        fp_ds = s["fp_signal"][::2]
        # Z-score
        fp_z = (fp_ds - fp_ds.mean()) / (fp_ds.std() + 1e-10)
        motifs = s["vame"]
        n = min(len(fp_z), len(motifs))
        fp_z = fp_z[:n]
        motifs = motifs[:n]

        # Find motif bout onsets
        changes = np.where(np.diff(motifs) != 0)[0] + 1
        for onset in changes:
            if onset - window_pre < 0 or onset + window_post > n:
                continue
            mid = motifs[onset]
            if mid in motif_triggered:
                snippet = fp_z[onset - window_pre:onset + window_post]
                motif_triggered[mid].append(snippet)

    # Build heatmap: rows = motifs (ordered by speed), columns = time
    time_axis = np.arange(-window_pre, window_post) / 30.0
    heatmap = np.full((len(motif_ids_sorted), window_pre + window_post), np.nan)
    for i, m in enumerate(motif_ids_sorted):
        if motif_triggered[m]:
            heatmap[i] = np.mean(motif_triggered[m], axis=0)

    fig, ax = plt.subplots(figsize=(8, 8))
    vmax = np.nanmax(np.abs(heatmap)) * 0.8
    im = ax.imshow(heatmap, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax,
                   extent=[time_axis[0], time_axis[-1], len(motif_ids_sorted) - 0.5, -0.5])
    ax.axvline(0, color="black", linewidth=1, linestyle="--")
    ax.set_xlabel("Time from motif onset (s)")
    ax.set_ylabel("Motif ID (ordered by speed)")
    ax.set_yticks(range(len(motif_ids_sorted)))
    ax.set_yticklabels(motif_ids_sorted, fontsize=6)
    plt.colorbar(im, ax=ax, label="z-scored FP signal")
    ax.set_title("Fig 5E: FP signal aligned to motif onset", fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("No sessions with both FP and VAME data available.")

## Figure 5H — FP AUC vs mean motif speed

For each motif, we compute the area under the curve (AUC) of the z-scored FP signal
in the 0–1 s window after motif onset and correlate it with mean motif speed.

In [ ]:
if fp_vame_sessions:
    auc_per_motif = []
    speed_per_motif_plot = []

    for m in motif_ids_sorted:
        if motif_triggered[m]:
            mean_trace = np.mean(motif_triggered[m], axis=0)
            # AUC from onset (index=window_pre) to 1s after (index=window_pre+30)
            auc = np.sum(mean_trace[window_pre:window_pre + 30]) / 30.0
            auc_per_motif.append(auc)
            speed_per_motif_plot.append(mean_speed_per_motif[m])

    auc_arr = np.array(auc_per_motif)
    speed_arr = np.array(speed_per_motif_plot)

    r, p = stats.pearsonr(speed_arr, auc_arr)

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(speed_arr, auc_arr, c="teal", s=40, edgecolors="black", linewidth=0.5)
    # Fit line
    slope, intercept = np.polyfit(speed_arr, auc_arr, 1)
    x_line = np.linspace(speed_arr.min(), speed_arr.max(), 100)
    ax.plot(x_line, slope * x_line + intercept, "k--", linewidth=1)
    ax.set_xlabel("Mean motif speed (pixels/s)")
    ax.set_ylabel("FP AUC (0–1 s)")
    ax.set_title(f"Fig 5H: FP AUC vs motif speed\nr={r:.2f}, p={p:.4f}", fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("No sessions with both FP and VAME data.")

## Figure 6B — Representative locomotor trajectories

Snout trajectories from DLC pose estimation for representative sessions from different
experimental groups, showing differences in locomotion patterns.

In [ ]:
# Map sessions to simplified group labels for Fig 6
# Use details.csv experiment field stored in session_id
fig6_groups = {
    "oft_Tetx": {"ctrl": "Ctrl-AAV", "Tetx": "Anxa1-TeTx"},
    "oft_6OHDA": {"asc_acid": "Ctrl-AA", "6OHDA": "6-OHDA-mild"},
    "oft_mitopark_10-11weeks": {"Ctrl 10-11 wks": "Ctrl MP-early", "KO 10-11 wks": "MP-early"},
    "oft_mitopark_15-18weeks": {"Ctrl 15-18 wks": "Ctrl MP-mid", "KO 15-18 wks": "MP-middle"},
}

# Get group label for each session from the NWB file name and session_id
import csv
details_path = Path("../../../nwb_output/open_field_test").parent.parent / "src/meletis_lab_to_nwb/open_field_test"
# We need to map NWB filenames back to their group. Let's use the details.csv directly.
data_dir = Path("/Volumes/T9/data/Meletis/oft")
with open(data_dir / "details.csv") as f:
    reader = csv.DictReader(f)
    details_map = {}
    for row in reader:
        details_map[row["mouse.ID"]] = details_map.get(row["mouse.ID"], {})
        details_map[row["mouse.ID"]][row["video"]] = row

# Annotate sessions with experiment and group
for s in sessions:
    sid = s["subject_id"]
    # Find matching row
    if sid in details_map:
        for vid, row in details_map[sid].items():
            if vid in s["name"]:
                s["raw_experiment"] = row["experiment"]
                s["raw_group"] = row["group"]
                break

# Select one session per group for trajectory plot
plot_groups = ["Ctrl-AAV", "Anxa1-TeTx", "6-OHDA-mild", "MP-early", "MP-middle"]
traj_sessions = []
for s in sessions:
    exp = s.get("raw_experiment", "")
    grp = s.get("raw_group", "")
    if exp in fig6_groups and grp in fig6_groups[exp]:
        label = fig6_groups[exp][grp]
        if label in plot_groups and not any(t[1] == label for t in traj_sessions):
            traj_sessions.append((s, label))

n_plots = len(traj_sessions)
fig, axes = plt.subplots(1, n_plots, figsize=(4 * n_plots, 4))
if n_plots == 1:
    axes = [axes]

for i, (s, label) in enumerate(traj_sessions):
    ax = axes[i]
    xy = s["snout_xy"]
    ax.plot(xy[:, 0], xy[:, 1], linewidth=0.2, alpha=0.6, color="black")
    ax.set_title(label, fontweight="bold")
    ax.set_aspect("equal")
    ax.set_xlabel("x (pixels)")
    ax.set_ylabel("y (pixels)")

plt.suptitle("Fig 6B: Locomotor trajectories", fontweight="bold", y=1.05)
plt.tight_layout()
plt.show()

## Figure 6C — Speed and angular speed box plots per group

Classical locomotor parameters compared across control and experimental groups.
Mean linear speed (top), mean angular speed (middle), and immobility time (bottom).

In [ ]:
# Compute per-session summary metrics
group_data = {}  # label -> list of (mean_speed, mean_angular_speed, immobility_frac)

for s in sessions:
    exp = s.get("raw_experiment", "")
    grp = s.get("raw_group", "")
    if exp in fig6_groups and grp in fig6_groups[exp]:
        label = fig6_groups[exp][grp]
    else:
        continue

    mean_speed = np.mean(s["speed"])
    mean_angular = np.mean(np.abs(s["angular_speed"]))
    # Immobility: fraction of time where speed < threshold
    immobility_frac = np.mean(s["speed"] < 2.0)  # threshold in pixels/s

    group_data.setdefault(label, []).append((mean_speed, mean_angular, immobility_frac))

# Define display order
display_order = ["Ctrl-AAV", "Anxa1-TeTx", "Ctrl-AA", "6-OHDA-mild",
                 "Ctrl MP-early", "MP-early", "Ctrl MP-mid", "MP-middle"]
display_order = [g for g in display_order if g in group_data]

# Colors: controls gray, experimentals colored
colors = {"Ctrl-AAV": "gray", "Anxa1-TeTx": "#E8963E",
          "Ctrl-AA": "gray", "6-OHDA-mild": "#D94E4E",
          "Ctrl MP-early": "gray", "MP-early": "#4EAED9",
          "Ctrl MP-mid": "gray", "MP-middle": "#4E6DD9"}

fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
metric_names = ["Mean linear speed (px/s)", "Mean angular speed (|rad/s|)", "Immobility fraction"]

for metric_idx, (ax, metric_name) in enumerate(zip(axes, metric_names)):
    positions = []
    data_lists = []
    color_list = []
    labels = []

    for i, g in enumerate(display_order):
        vals = [d[metric_idx] for d in group_data[g]]
        positions.append(i)
        data_lists.append(vals)
        color_list.append(colors.get(g, "gray"))
        labels.append(g)

    bp = ax.boxplot(data_lists, positions=positions, widths=0.6, patch_artist=True)
    for patch, color in zip(bp["boxes"], color_list):
        patch.set_facecolor(color)
        patch.set_alpha(0.5)

    # Overlay individual points
    for i, (vals, color) in enumerate(zip(data_lists, color_list)):
        jitter = np.random.normal(0, 0.05, len(vals))
        ax.scatter(np.full(len(vals), i) + jitter, vals, c=color, s=30, edgecolors="black",
                   linewidth=0.5, zorder=5)

    ax.set_ylabel(metric_name)
    ax.set_xticks(positions)
    ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=8)

axes[0].set_title("Fig 6C: Classical locomotor parameters", fontweight="bold")
plt.tight_layout()
plt.show()

## Figure 6I — Normalized motif usage across groups

For each experimental group, we compute the fraction of time spent in each VAME motif
(normalized by total frames), ordered by increasing mean motif speed.

In [ ]:
# Compute motif usage per group
group_usage = {}  # label -> array of motif fractions (averaged across sessions)

for s in sessions:
    if s["vame"] is None:
        continue
    exp = s.get("raw_experiment", "")
    grp = s.get("raw_group", "")
    if exp in fig6_groups and grp in fig6_groups[exp]:
        label = fig6_groups[exp][grp]
    else:
        continue

    motifs = s["vame"]
    usage = np.zeros(len(motif_ids_sorted))
    for i, m in enumerate(motif_ids_sorted):
        usage[i] = np.sum(motifs == m) / len(motifs)

    group_usage.setdefault(label, []).append(usage)

# Plot
plot_labels = [g for g in display_order if g in group_usage]

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(motif_ids_sorted))
width = 0.8 / len(plot_labels)

for i, label in enumerate(plot_labels):
    usage_arr = np.array(group_usage[label])
    mean_usage = usage_arr.mean(axis=0)
    sem_usage = usage_arr.std(axis=0) / np.sqrt(len(usage_arr)) if len(usage_arr) > 1 else np.zeros_like(mean_usage)
    offset = (i - len(plot_labels) / 2 + 0.5) * width
    ax.bar(x + offset, mean_usage, width, yerr=sem_usage,
           label=label, color=colors.get(label, "gray"), alpha=0.7,
           edgecolor="black", linewidth=0.3, capsize=1)

ax.set_xticks(x)
ax.set_xticklabels(motif_ids_sorted, fontsize=6, rotation=90)
ax.set_xlabel("Motif ID (ordered by speed)")
ax.set_ylabel("Fraction of time")
ax.set_title("Fig 6I: Normalized motif usage across groups", fontweight="bold")
ax.legend(fontsize=7, ncol=2, loc="upper left")
plt.tight_layout()
plt.show()

## Cleanup

In [ ]:
for s in sessions:
    s["io"].close()